# Notebook 02 — Experiments (U-Net, Siamese, ablations)

**Team:** BuildSafe Vision (DATA 255, Fall 2026)  
**Depends on:** notebook 01 manifest for `benchmark` / `full` modes (not required for `smoke`).  
**Papers:** see `docs/RELATED_WORK.md` (Daudt triad + LEVIR metrics; no STANet attention).

## What we implemented
| Piece | Location | Notes |
| --- | --- | --- |
| FC-EF (6-ch U-Net) | `models.py` | Daudt early fusion; prior val F1 0.7319 |
| FC-Siam-diff | `models.py` | Shared encoder + `|fA−fB|` (proposal H1) |
| FC-Siam-conc | `models.py` | Shared encoder + concat skips (completes Daudt triad) |
| BCE / BCE+Dice | `losses.py` | Ablation 1 |
| Paired geometric aug | `data.py` | Ablation 2 |
| Abs-diff baseline | `baseline.py` | Non-learned control |
| Pooled + per-image metrics | `metrics.py` | Chen/Shi change-class scores |
| Train loop + latency | `train.py` | Checkpoint by validation F1 |
| Smoke path | `scripts/run_smoke.py` | Synthetic tiles |

## Run modes
| `RUN_MODE` | Data | Typical use |
| --- | --- | --- |
| `smoke` | Synthetic tiles | After coding — no LEVIR download |
| `benchmark` | Full train/val, 1 epoch | Timing / device sanity |
| `full` | Full train/val, ≤30 epochs | Real comparison / ablations |

## Safety rules
- Primary metric: **changed-class F1** (not accuracy).
- Threshold fixed at **0.5** unless a validation-only study is predeclared.
- `ALLOW_TEST` stays `False` until model choices are frozen.



## 1. Imports and environment

Adds `src/` to `sys.path` so the notebook can import `levir_cd` whether you launched Jupyter from the repo root or from `notebooks/`.


In [ ]:
import os
import sys
from pathlib import Path

# Resolve project root, then expose the local package.
_here = Path.cwd().resolve()
_project = _here.parent if _here.name == "notebooks" else _here
_project = Path(os.environ.get("LEVIR_PROJECT_ROOT", str(_project))).expanduser().resolve()
sys.path.insert(0, str(_project / "src"))

from levir_cd.baseline import absolute_difference_baseline
from levir_cd.config import ExperimentConfig
from levir_cd.data import ChangeTiles, load_split_frames, make_loaders
from levir_cd.device import describe_runtime, pick_device, set_seed
from levir_cd.models import build_model, count_parameters
from levir_cd.train import maybe_copy_summary_to_reports, run_experiment

print("Project root:", _project)
print("Package import OK")


## 2. Experiment configuration

Change the knobs below between runs. Defaults start in **`smoke`** so a fresh clone does not accidentally launch a multi-hour training job.


In [ ]:
cfg = ExperimentConfig(
    PROJECT_ROOT=str(_project),
    # --- choose one run mode ---
    RUN_MODE="smoke",          # "smoke" | "benchmark" | "full"
    # --- Daudt triad / ablations ---
    MODEL="unet",              # "unet" | "siamese_diff" | "siamese_conc"
    LOSS="bce_dice",           # "bce" | "bce_dice"
    AUGMENT=True,              # paired flips + 90° rotations
    SEED=1511,                 # also try 42 and 123 for the main comparison
    # --- training defaults for real LEVIR tiles ---
    TILE_SIZE=256,
    IMAGE_SIZE=1024,
    BATCH_SIZE=4,
    BASE_CHANNELS=16,
    LEARNING_RATE=1e-3,
    WEIGHT_DECAY=1e-4,
    FULL_EPOCHS=30,
    EARLY_STOP_PATIENCE=8,
    MIN_EPOCHS=12,
    THRESHOLD=0.5,
    # --- smoke-only size knobs (ignored in benchmark/full) ---
    SMOKE_TRAIN_TILES=8,
    SMOKE_VAL_TILES=4,
    SMOKE_EPOCHS=1,
    SMOKE_BATCH_SIZE=2,
)

# Faster CPU smoke defaults. Real LEVIR runs keep 256 / base-16 above.
if cfg.RUN_MODE == "smoke":
    cfg.TILE_SIZE = 64
    cfg.BASE_CHANNELS = 8

cfg.validate()
device = pick_device()
set_seed(cfg.SEED)

print(describe_runtime(device))
print("Run name :", cfg.run_name)
print("Manifest :", cfg.manifest_path)
print("Output   :", cfg.run_dir)
print("Epochs   :", cfg.effective_epochs(), "| batch:", cfg.effective_batch_size())



## 3. Reference baseline — absolute RGB difference

Non-learned control from the proposal. On validation tiles we pick a threshold that maximizes changed-class F1. Skipped in `smoke` because there is no LEVIR manifest/imagery yet.


In [ ]:
baseline_result = None

if cfg.RUN_MODE == "smoke":
    print("Smoke mode: skip abs-diff baseline (synthetic data only).")
else:
    # Requires notebook 01 to have written split_manifest.csv on this machine.
    assert cfg.manifest_path.is_file(), (
        "Missing manifest. Run notebooks/01_dataset_audit.ipynb first."
    )
    train_df, val_df = load_split_frames(cfg.manifest_path)
    val_ds = ChangeTiles(
        val_df,
        size=cfg.TILE_SIZE,
        image_size=cfg.IMAGE_SIZE,
        augment=False,
    )
    # Reuse make_loaders helper; we only need the validation iterator.
    _, val_loader = make_loaders(val_ds, val_ds, batch_size=cfg.BATCH_SIZE)
    baseline_result = absolute_difference_baseline(val_loader)
    print("Absolute-difference baseline (validation-selected threshold):")
    for key, value in baseline_result.items():
        print(f"  {key}: {value}")


## 4. Model summary (before training)

Quick parameter count and a dry forward pass shape check. Both models consume the same 6-channel dataloader tensor; Siamese splits channels inside the network.


In [ ]:
import torch

model_preview = build_model(cfg.MODEL, base_channels=cfg.BASE_CHANNELS).to(device)
n_params = count_parameters(model_preview)
print(f"Model: {cfg.MODEL}")
print(f"Trainable parameters: {n_params:,}")

with torch.no_grad():
    demo = model_preview(
        torch.zeros(1, 6, cfg.TILE_SIZE, cfg.TILE_SIZE, device=device)
    )
print("Forward shape:", tuple(demo.shape))
assert tuple(demo.shape) == (1, 1, cfg.TILE_SIZE, cfg.TILE_SIZE)
del model_preview, demo


## 5. Train and validate

`run_experiment` owns the matched protocol: same tiles/metrics/checkpoint rule for U-Net and Siamese. Artifacts land under `results/experiments/<run_name>/`:

- `config.json` — frozen knobs for this run  
- `metrics.csv` — per-epoch train/val curves  
- `best_model.pt` / `last_model.pt` — checkpoints (gitignored)  
- `summary.json` — F1, params, timing, batch-1 latency  

Smoke/benchmark summaries are also copied under `reports/experiments/` for quick sharing.


In [ ]:
summary = run_experiment(cfg, device=device)
report_dir = maybe_copy_summary_to_reports(summary, Path(cfg.PROJECT_ROOT))

print("\n=== Run summary ===")
for key in (
    "run_mode",
    "model",
    "loss",
    "augment",
    "seed",
    "best_epoch",
    "best_val_f1",
    "parameters",
    "batch_one_inference_seconds",
    "test_set_touched",
    "run_dir",
):
    print(f"{key}: {summary.get(key)}")
print("Report copy:", report_dir)


## 6. Read learning curves (when metrics exist)

For `smoke`, the CSV has a single epoch — useful as a pipeline check, not a quality claim. For `full`, compare train/val loss and validation F1/IoU across epochs.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = Path(summary["run_dir"]) / "metrics.csv"
history = pd.read_csv(metrics_path)
print(history.tail())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.epoch, history.train_loss, "o-", label="Train loss")
axes[0].plot(history.epoch, history.val_loss, "o-", label="Val loss")
axes[0].set(title=f"Loss ({cfg.LOSS})", xlabel="Epoch", ylabel="Loss")
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].plot(history.epoch, history.val_f1, "o-", label="Val F1")
axes[1].plot(history.epoch, history.val_iou, "o-", label="Val IoU")
axes[1].set(title=f"{cfg.MODEL} change-pixel scores", xlabel="Epoch", ylabel="Score", ylim=(0, 1))
axes[1].legend()
axes[1].grid(alpha=0.2)

fig.tight_layout()
curve_path = Path(summary["run_dir"]) / "learning_curves.png"
fig.savefig(curve_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", curve_path)


## 7. Ablation and multi-seed checklist

Re-run **Section 2 → Section 5** with one change at a time. Do not peek at test data while selecting the winner.

### A. Main comparison (proposal H1)
1. `MODEL="unet"` (FC-EF), `LOSS="bce_dice"`, `AUGMENT=True`, seeds `1511`, `42`, `123`
2. `MODEL="siamese_diff"` (FC-Siam-diff) with the same loss/aug/seeds
3. Optional: `MODEL="siamese_conc"` (FC-Siam-conc) under the same protocol
4. Report mean ± std validation F1 / IoU / precision / recall, plus params and latency

### B. Loss ablation (proposal H2)
- Chosen architecture, `LOSS="bce"` vs `LOSS="bce_dice"`, other knobs fixed

### C. Paired augmentation ablation
- Chosen architecture, `AUGMENT=True` vs `AUGMENT=False`

### D. After freezing choices
- Locked **one-shot** test evaluation outside exploratory loops
- Use `per_image_val_metrics.csv` for missed buildings / false positives / weak boundaries
- Final write-up in `report/report.md`



In [ ]:
checklist = [
    ("Main", "unet vs siamese_diff (+ optional siamese_conc)", "LOSS=bce_dice, AUGMENT=True, seeds 1511/42/123"),
    ("Loss", "bce vs bce_dice", "same model/aug/seed"),
    ("Augmentation", "AUGMENT True vs False", "same model/loss/seed"),
    ("Efficiency", "params / train time / batch-1 latency", "logged in summary.json"),
    ("Errors", "per_image_val_metrics.csv", "worst/best F1 tiles on validation"),
    ("Test", "single locked eval", "only after freezing all choices"),
]
print("Remaining experiment matrix:")
for name, change, hold in checklist:
    print(f"- {name}: change [{change}]; hold [{hold}]")

print("\nCLI smoke (Daudt triad, no Jupyter):")
print("  PYTHONPATH=src python scripts/run_smoke.py --model all")
print("Unit tests:")
print("  PYTHONPATH=src python -m pytest -q")



## 8. Interpreting results (short)

| Observation | Likely meaning |
| --- | --- |
| Smoke F1 near 0 | Expected on tiny synthetic 1-epoch runs — pipeline check only |
| Benchmark finishes one epoch | Device/memory OK; not a quality claim |
| Full val F1 competitive with 0.73 | Comparable to the earlier U-Net feasibility run |
| Siamese helps F1 but is slower | Supports H1 quality claim with a compute trade-off |
| BCE+Dice lifts recall/F1 vs BCE | Supports H2 under class imbalance |

Keep every published table tied to a `summary.json` + seed + `run_name`.
